# 05. Naive Bayes - Implementation, training và testing

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/ml_scratch').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Mở notebook trong dự án')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import display, Image


## Mục đích
mục đích: Với một email ham hoặc spam đã thành vector TF-IDF, prior và các từ trong thư đẩy điểm log về lớp nào?

Input cùng ma trận bước 03. `fit` tính `class_prior` và `laplace_probabilities` trên train, alpha=1. `log_score` của email mới là log prior cộng tổng tfidf nhân log P(từ|lớp). Lớp điểm cao hơn được chọn.

Ô tiếp theo fit trên train thật, rồi in prior, tám từ nặng nhất của một ham và một spam trong test, và điểm log của chính hai thư đó. Ví dụ hai từ nằm ở `docs/Algorithms/naive_bayes.md`. Không chọn alpha theo test.

## Một ham và một spam trong test
Sau `fit`, mỗi lớp có prior và một xác suất cho từng từ trong 1.000 chiều. Bảng từ lấy tám thành phần TF-IDF lớn nhất của email test. Cột `P_word_given_ham` và `P_word_given_spam` là Laplace đã học. Bảng điểm là `log_score` của cả vector, không chỉ của tám từ, rồi `normalized_scores`.

In [ ]:
from ml_scratch.data import load_features
from ml_scratch.classifiers import NaiveBayesSpamClassifier
from ml_scratch.evaluation import explain_nb

X_train, X_test, train, test, vectorizer = load_features()
print('Train', X_train.shape, 'Test', X_test.shape)
model = NaiveBayesSpamClassifier(alpha=1)
model.build_model()
model.fit(X_train, train.label)
print('Prior', {label: float(model.prior[label]) for label in model.classes})

for label in ['ham', 'spam']:
    position = test.label.to_numpy().tolist().index(label)
    row = test.iloc[position]
    print('---', label, 'source_row', row.name)
    print(row.clean_text[:240])
    terms, scores = explain_nb(model, vectorizer, X_test[position])
    display(terms)
    display(scores)
    print('Predict', model.predict(X_test[position:position+1])[0])

## Chấm toàn bộ tập test
Ô dưới gọi `fit_evaluate("naive_bayes")` trên cùng 977 email. `fit` tính `class_prior` và `laplace_probabilities` trên train, alpha=1. `log_score` của một email là log prior cộng tổng tfidf nhân log P(từ|lớp). Lớp có điểm cao hơn được chọn. Spam score đưa vào ROC là thành phần spam của `normalized_scores`, không phải xác suất đã calibration.

Alpha=1 cố định từ trước. Không chọn alpha theo ô này.

Ô ghi `artifacts/naive_bayes.json`, bốn CSV tiền tố `naive_bayes_` và `outputs/figures/naive_bayes_test.png`. `examples()` vẽ `naive_bayes_example.png` cho ví dụ hai từ, không phải bảng fit Enron ở ô trên.

In [ ]:
from ml_scratch.evaluation import fit_evaluate, examples
metrics = fit_evaluate("naive_bayes")
examples()
metrics

## Đọc hai hình
`naive_bayes_test.png` là confusion matrix và ROC của score đã chuẩn hóa, trên cùng 685 ham và 292 spam.

`naive_bayes_example.png` minh họa prior, Laplace và log score với hai từ. Không đọc hình này như kết quả trên tập test.

In [ ]:
display(Image(filename=str(ROOT/"outputs/figures/naive_bayes_test.png")))
display(Image(filename=str(ROOT/"outputs/figures/naive_bayes_example.png")))

## Đọc số và handoff
Trên cùng 977 email, Naive Bayes có TN 632, FP 53, FN 18, TP 274. Accuracy 92,73%, precision 83,79%, recall 93,84%, F1 88,53%, FPR 7,74%, AUC 0,9811. So với KNN trên split này, model chặn nhầm nhiều ham hơn (FP 53 thay vì 20) và sót thêm spam (FN 18 thay vì 15). `naive_bayes_test.png` là confusion matrix và ROC của score đã chuẩn hóa.

Bảng từ và điểm log ở trên lấy từ email test thật. Hình `naive_bayes_example.png` chỉ là phác họa hai từ, không phải bảng fit Enron.

Output: `artifacts/naive_bayes.json`; bốn CSV tiền tố `naive_bayes_` trong `outputs/`. Bước 06 so file dự đoán này với KNN trên cùng `source_row`.